# HITO 2

a reproducible training and evaluation notebook covering both targets. The notebook should:

>   Load the same dataset as Hito 1

 >   Use the same locked temporal split (train 2019–2021 / calibration 2022 / test 2023–2024)
 
 >   Train and evaluate models for both targets

  >  Produce calibrated probability output for binary targets, and a probability-quality analysis for regression targets


## Summary and purpose

This notebook implements a reproducible modeling pipeline for Hito 2 using the same dataset and locked temporal split used in Hito 1. It provides: data loading, the locked temporal split (train 2019–2021 / calibration 2022 / test 2023–2024), simple baseline training for both targets, probability calibration for binary targets, and evaluation guidance.


In [4]:
# Setup: imports and data path
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss, roc_auc_score, mean_squared_error

# Data directory: Capstone/data relative to this notebook
DATA_DIR = Path('..') / 'data'
RACE_CSV = DATA_DIR / 'f1_strategy_race_level.csv'
LAP_CSV = DATA_DIR / 'f1_strategy_lap_level.csv'
print('Data files present:', RACE_CSV.exists(), LAP_CSV.exists())


Data files present: True True


In [5]:
# Load race-level data (fallback to lap-level if needed)
if RACE_CSV.exists():
    df = pd.read_csv(RACE_CSV)
else:
    df = pd.read_csv(LAP_CSV)

print('Rows, cols:', df.shape)
print('Columns sample:', list(df.columns)[:20])


Rows, cols: (2447, 47)
Columns sample: ['season', 'round', 'race_name', 'circuit_id', 'circuit', 'circuit_type', 'driver_id', 'driver_name', 'Driver', 'Team', 'constructor_name', 'grid_position', 'qualifying_position', 'qualifying_time_s', 'driver_prior3_avg_finish', 'constructor_prior3_avg_finish', 'driver_circuit_prior_avg', 'constructor_tier', 'n_stops', 'strategy_type']


## Locked temporal split

We use the locked split requested by the prompt: train seasons 2019–2021, calibration 2022, test 2023–2024. Ensure your data has a `season` column; otherwise adapt accordingly.


In [6]:
# Create temporal splits (adjust column name if needed)
assert 'season' in df.columns, 'Dataset must include a season column for temporal split'
train_df = df[df['season'].isin([2019,2020,2021])].copy()
cal_df = df[df['season']==2022].copy()
test_df = df[df['season'].isin([2023,2024])].copy()
print('train, cal, test sizes:', len(train_df), len(cal_df), len(test_df))


train, cal, test sizes: 1132 426 889


## Targets

This notebook expects two targets present or derivable in the dataset: `is_top10` (binary) and an expansion target (regression or probability). If column names differ, adapt the cell below.


In [7]:
# Check for target columns and create placeholders if missing
targets = {}
if 'is_top10' in df.columns:
    targets['is_top10'] = 'is_top10'
else:
    print('Warning: is_top10 not found; please create this binary target')

# Example expansion target name: 'expansion_target'
if 'expansion_target' in df.columns:
    targets['expansion'] = 'expansion_target'
else:
    print('Warning: expansion_target not found; create or rename accordingly')

targets


{'is_top10': 'is_top10'}

## Modeling scaffold (baseline)

Below are simple baseline training examples: logistic regression for `is_top10` with calibration, and a ridge regression placeholder for the expansion target. Replace feature selection with your engineered features and add preprocessing as needed.


In [10]:
# Select numeric feature set for a simple baseline (users should replace this with engineered features)
numeric_cols = list(train_df.select_dtypes(include=[np.number]).columns)
feature_cols = [c for c in numeric_cols if c not in ['season','raceId','is_top10','expansion_target']]
feature_cols = feature_cols[:20]
print('Using numeric features:', feature_cols)

# Prepare data for is_top10 if present
if 'is_top10' in targets.values():
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df['is_top10']
    X_cal = cal_df[feature_cols].fillna(0)
    y_cal = cal_df['is_top10']
    X_test = test_df[feature_cols].fillna(0)
    y_test = test_df['is_top10']

    base_clf = LogisticRegression(max_iter=1000)
    calib = CalibratedClassifierCV(estimator=base_clf, cv='prefit')
    # fit on train, calibrate on calibration set
    base_clf.fit(X_train, y_train)
    calib.fit(X_cal, y_cal)
    p_test = calib.predict_proba(X_test)[:,1]
    print('Brier (is_top10):', brier_score_loss(y_test, p_test))
    print('AUC (is_top10):', roc_auc_score(y_test, p_test))

# Expansion target baseline (regression)
if 'expansion' in targets:
    if 'expansion_target' in train_df.columns:
        Xr_train = train_df[feature_cols].fillna(0)
        yr_train = train_df['expansion_target']
        Xr_test = test_df[feature_cols].fillna(0)
        yr_test = test_df['expansion_target']
        reg = Ridge()
        reg.fit(Xr_train, yr_train)
        yp = reg.predict(Xr_test)
        print('MSE (expansion):', mean_squared_error(yr_test, yp))
    else:
        print('No expansion_target column in data; skip regression baseline')


Using numeric features: ['round', 'grid_position', 'qualifying_position', 'qualifying_time_s', 'driver_prior3_avg_finish', 'constructor_prior3_avg_finish', 'driver_circuit_prior_avg', 'n_stops', 'stint1_length', 'stint2_length', 'stint3_length', 'stint4_length', 'stint5_length', 'avg_pit_stop_duration_s', 'total_pit_time_s', 'first_pit_lap', 'last_pit_lap', 'safety_car_periods', 'safety_car_laps', 'vsc_laps']
Brier (is_top10): 0.12605949211485018
AUC (is_top10): 0.9012567143001926


c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


## Evaluation and calibration notes

- Report Brier score and AUC for `is_top10` on the test split.
- For the expansion target, report MSE and provide a probability-quality analysis if the target is probabilistic.
- Run the ablation tests described in `leakage_audit.md` to quantify the role of strategy features.


## Feature provenance and leakage checks (follow-up)

- Annotate each engineered feature with its source and timestamp relative to the decision point.
- Add a short notebook cell which re-trains models with strategy features removed to measure information leakage (see `leakage_audit.md`).


## Step 6 — Interpretation (write this in your team's words)

Fill in the four placeholders below with concrete numbers from your run. **Use scenario-conditioned language**, not causal language. Then commit this notebook to your repo.

> **Driver-race context (held fixed):** [driver, circuit, season, grid position, prior form summary]
>
> **Strategy inputs varied:** ["n_stops" "strategy_type",
            "compound_sequence"
            "stint1_length" "stint2_length" "stint3_length",
            "avg_pit_stop_duration_s", "total_pit_time_s",
            "first_pit_lap", "last_pit_lap"]
>
> **Result:** Under our model and our 2019–2022 training distribution, holding the driver-race context fixed:
> - For target `is_top10`: P(top10 | A) = 0.927681 vs P(top10 | B) = 0.882214 → preferred scenario: Hito 1 1-stop
> - For target `is_top5`: P(is_top5 | A) = 0.828151 vs P(is_top5 | B) = 0.750936 → preferred scenario: Hito 1 1-stop
>
> **The two targets AGREE on the recommended scenario.**
>
> **What this means for our advisor:** 
The 1-stop strategy is preferred not just for securing a top-10 finish, but even more strongly for a top-5 finish. This magnified preference at the sharper threshold—which is_top10 alone would never reveal—suggests the pit-stop plan is particularly robust for drivers fighting for podium positions rather than merely points-scoring spots. Under our training distribution and holding context fixed, the strategy desk can be more confident in this 1-stop bet when the race target shifts upward.
>
> **What we still don't know:** 
In our training data, pit-stop strategy is not chosen at random—teams selected 1-stop vs. 2-stop based on live car performance, fuel consumption, tire wear, and race conditions (e.g., safety car timing) that the model never observes. Our what-if comparison cannot separate whether the 1-stop advantage comes from the strategy itself or from the unobserved pace and conditions that made the strategy desk confident enough to deploy it. 

---

